In [1]:
import os

print(os.listdir("../data/processed"))

['category_translation_clean.csv', 'churn_model_comparison.csv', 'clv_contribution.csv', 'clv_summary.csv', 'customers_clean.csv', 'customer_360.csv', 'customer_360_churn.csv', 'customer_360_feature_engineered.csv', 'customer_churn.csv', 'customer_clv.csv', 'geolocation_clean.csv', 'orders_clean.csv', 'order_items_clean.csv', 'payments_clean.csv', 'products_clean.csv', 'reviews_clean.csv', 'sellers_clean.csv']


In [2]:
import pandas as pd
import numpy as np

orders = pd.read_csv(
    "../data/processed/orders_clean.csv"
)

customer_360 = pd.read_csv(
    "../data/processed/customer_360.csv"
)

print("Orders shape:", orders.shape)
print("Customer 360 shape:", customer_360.shape)

Orders shape: (99441, 12)
Customer 360 shape: (96096, 47)


In [3]:
print(orders.columns.tolist())

['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'delivery_days', 'estimated_delivery_days', 'delivery_delay_days', 'delivery_performance']


In [4]:
orders["order_purchase_timestamp"] = pd.to_datetime(
    orders["order_purchase_timestamp"]
)

print(
    "First order:",
    orders["order_purchase_timestamp"].min()
)

print(
    "Last order:",
    orders["order_purchase_timestamp"].max()
)

First order: 2016-09-04 21:15:19
Last order: 2018-10-17 17:30:18


In [5]:
cutoff_date = pd.Timestamp("2018-01-01")

observation_orders = orders[
    orders["order_purchase_timestamp"] < cutoff_date
].copy()

future_orders = orders[
    orders["order_purchase_timestamp"] >= cutoff_date
].copy()

print("Observation orders:", observation_orders.shape)
print("Future orders:", future_orders.shape)

print(
    "Future period:",
    future_orders["order_purchase_timestamp"].min(),
    "→",
    future_orders["order_purchase_timestamp"].max()
)

Observation orders: (45430, 12)
Future orders: (54011, 12)
Future period: 2018-01-01 02:48:41 → 2018-10-17 17:30:18


In [6]:
print("customer_unique_id" in orders.columns)

False


In [7]:
customers = pd.read_csv(
    "../data/processed/customers_clean.csv"
)

print(customers.shape)
print(customers.columns.tolist())

(99441, 5)
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']


In [8]:
orders = orders.merge(
    customers[
        ["customer_id", "customer_unique_id"]
    ],
    on="customer_id",
    how="left"
)

print("Orders shape:", orders.shape)
print(
    "Missing customer_unique_id:",
    orders["customer_unique_id"].isna().sum()
)

Orders shape: (99441, 13)
Missing customer_unique_id: 0


In [9]:
# Recreate the time periods

cutoff_date = pd.Timestamp("2018-01-01")

observation_orders = orders[
    orders["order_purchase_timestamp"] < cutoff_date
].copy()

future_orders = orders[
    orders["order_purchase_timestamp"] >= cutoff_date
].copy()

print("Observation orders:", observation_orders.shape)
print("Future orders:", future_orders.shape)

Observation orders: (45430, 13)
Future orders: (54011, 13)


In [10]:
# Find repeat customers
observation_customers = set(
    observation_orders["customer_unique_id"]
    .dropna()
    .unique()
)

future_customers = set(
    future_orders["customer_unique_id"]
    .dropna()
    .unique()
)

repeat_customers = (
    observation_customers.intersection(future_customers)
)

print("Observation customers:", len(observation_customers))
print("Future customers:", len(future_customers))
print("Repeat customers:", len(repeat_customers))

Observation customers: 44034
Future customers: 52749
Repeat customers: 687


In [11]:
# Create the target variable

repeat_data = pd.DataFrame({
    "customer_unique_id": list(observation_customers)
})

repeat_data["repeat_purchase"] = (
    repeat_data["customer_unique_id"]
    .isin(repeat_customers)
    .astype(int)
)

print(repeat_data["repeat_purchase"].value_counts())
print(
    repeat_data["repeat_purchase"]
    .value_counts(normalize=True) * 100
)

repeat_purchase
0    43347
1      687
Name: count, dtype: int64
repeat_purchase
0    98.439842
1     1.560158
Name: proportion, dtype: float64


In [12]:
# Add customer features
repeat_data = repeat_data.merge(
    customer_360,
    on="customer_unique_id",
    how="left"
)

print("Repeat dataset shape:", repeat_data.shape)
print("Missing customer features:", repeat_data.isna().sum().sum())

Repeat dataset shape: (44034, 48)
Missing customer features: 5209


In [13]:
print(
    repeat_data[
        [
            "customer_unique_id",
            "repeat_purchase",
            "total_orders",
            "total_spent",
            "avg_order_value",
            "avg_review_score"
        ]
    ].head()
)

                 customer_unique_id  repeat_purchase  total_orders  \
0  b99d74a79c10e715adaf03cd5779b787                0             1   
1  a84c2369d0550329ec9509ed225e210f                0             1   
2  bd3410fa50b021fbc967037cf14144de                0             1   
3  194a47bc9dd220ae9e221d44aced5fab                0             1   
4  3894c7d8e29573e308ebc285ecd61d39                0             1   

   total_spent  avg_order_value  avg_review_score  
0       204.08           204.08               5.0  
1        54.09            54.09               5.0  
2        86.20            86.20               1.0  
3        35.41            35.41               5.0  
4       294.14           294.14               4.0  


In [14]:
print(observation_orders.columns.tolist())

['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'delivery_days', 'estimated_delivery_days', 'delivery_delay_days', 'delivery_performance', 'customer_unique_id']


In [15]:
# reate observation-period customer features
observation_features = (
    observation_orders
    .groupby("customer_unique_id")
    .agg(
        total_orders_obs=("order_id", "nunique"),
        first_purchase_date=("order_purchase_timestamp", "min"),
        last_purchase_date=("order_purchase_timestamp", "max")
    )
    .reset_index()
)

# Customer lifetime in days during observation period
observation_features["customer_lifetime_days_obs"] = (
    observation_features["last_purchase_date"]
    - observation_features["first_purchase_date"]
).dt.days

# Make sure single-order customers don't have zero lifetime
observation_features["customer_lifetime_days_obs"] = (
    observation_features["customer_lifetime_days_obs"].fillna(0)
)

print(observation_features.shape)
print(observation_features.head())

(44034, 5)
                 customer_unique_id  total_orders_obs first_purchase_date  \
0  0000f46a3911fa3c0805444483337064                 1 2017-03-10 21:05:03   
1  0000f6ccb0745a6a4b88665a16c9f078                 1 2017-10-12 20:29:41   
2  0004aac84e0df4da2b147fca70cf8255                 1 2017-11-14 19:45:42   
3  0005e1862207bf6ccc02e4228effd9a0                 1 2017-03-04 23:32:12   
4  0006fdc98a402fceb4eb0ee528f6a8d4                 1 2017-07-18 09:23:10   

   last_purchase_date  customer_lifetime_days_obs  
0 2017-03-10 21:05:03                           0  
1 2017-10-12 20:29:41                           0  
2 2017-11-14 19:45:42                           0  
3 2017-03-04 23:32:12                           0  
4 2017-07-18 09:23:10                           0  


In [16]:
# Add spending features
order_items = pd.read_csv(
    "../data/processed/order_items_clean.csv"
)

print("Order items shape:", order_items.shape)
print(order_items.columns.tolist())

Order items shape: (112650, 8)
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value', 'item_total']


In [17]:
observation_items = observation_orders.merge(
    order_items[
        ["order_id", "price", "freight_value"]
    ],
    on="order_id",
    how="left"
)

print("Observation items shape:", observation_items.shape)

Observation items shape: (51773, 15)


In [18]:
# ustomer spending:
spending_features = (
    observation_items
    .groupby("customer_unique_id")
    .agg(
        total_spent_obs=("price", "sum"),
        total_freight_obs=("freight_value", "sum"),
        avg_order_value_obs=("price", "mean")
    )
    .reset_index()
)

print(spending_features.head())



                 customer_unique_id  total_spent_obs  total_freight_obs  \
0  0000f46a3911fa3c0805444483337064            69.00              17.22   
1  0000f6ccb0745a6a4b88665a16c9f078            25.99              17.63   
2  0004aac84e0df4da2b147fca70cf8255           180.00              16.89   
3  0005e1862207bf6ccc02e4228effd9a0           135.00              15.12   
4  0006fdc98a402fceb4eb0ee528f6a8d4            13.90              15.10   

   avg_order_value_obs  
0                69.00  
1                25.99  
2               180.00  
3               135.00  
4                13.90  


In [19]:
observation_features = observation_features.merge(
    spending_features,
    on="customer_unique_id",
    how="left"
)

print(observation_features.shape)
print(observation_features.head())

(44034, 8)
                 customer_unique_id  total_orders_obs first_purchase_date  \
0  0000f46a3911fa3c0805444483337064                 1 2017-03-10 21:05:03   
1  0000f6ccb0745a6a4b88665a16c9f078                 1 2017-10-12 20:29:41   
2  0004aac84e0df4da2b147fca70cf8255                 1 2017-11-14 19:45:42   
3  0005e1862207bf6ccc02e4228effd9a0                 1 2017-03-04 23:32:12   
4  0006fdc98a402fceb4eb0ee528f6a8d4                 1 2017-07-18 09:23:10   

   last_purchase_date  customer_lifetime_days_obs  total_spent_obs  \
0 2017-03-10 21:05:03                           0            69.00   
1 2017-10-12 20:29:41                           0            25.99   
2 2017-11-14 19:45:42                           0           180.00   
3 2017-03-04 23:32:12                           0           135.00   
4 2017-07-18 09:23:10                           0            13.90   

   total_freight_obs  avg_order_value_obs  
0              17.22                69.00  
1              17

In [20]:
cutoff_date = pd.Timestamp("2018-01-01")

observation_features["recency_days"] = (
    cutoff_date -
    observation_features["last_purchase_date"]
).dt.days

observation_features["purchase_frequency_obs"] = (
    observation_features["total_orders_obs"] /
    (observation_features["customer_lifetime_days_obs"] / 30 + 1)
)

print(
    observation_features[
        [
            "customer_unique_id",
            "total_orders_obs",
            "total_spent_obs",
            "recency_days",
            "purchase_frequency_obs"
        ]
    ].head(10)
)

                 customer_unique_id  total_orders_obs  total_spent_obs  \
0  0000f46a3911fa3c0805444483337064                 1            69.00   
1  0000f6ccb0745a6a4b88665a16c9f078                 1            25.99   
2  0004aac84e0df4da2b147fca70cf8255                 1           180.00   
3  0005e1862207bf6ccc02e4228effd9a0                 1           135.00   
4  0006fdc98a402fceb4eb0ee528f6a8d4                 1            13.90   
5  00082cbe03e478190aadbea78542e933                 1            79.00   
6  000a5ad9c4601d2bbdd9ed765d5213b3                 1            76.99   
7  000bfa1d2f1a41876493be685390d6d3                 1            35.00   
8  000c8bdb58a29e7115cfc257230fb21b                 1            13.90   
9  000de6019bb59f34c099a907c151d855                 1           229.80   

   recency_days  purchase_frequency_obs  
0           296                     1.0  
1            80                     1.0  
2            47                     1.0  
3           302  

In [22]:
observation_items.columns.tolist()

['order_id',
 'customer_id',
 'order_status',
 'order_purchase_timestamp',
 'order_approved_at',
 'order_delivered_carrier_date',
 'order_delivered_customer_date',
 'order_estimated_delivery_date',
 'delivery_days',
 'estimated_delivery_days',
 'delivery_delay_days',
 'delivery_performance',
 'customer_unique_id',
 'price',
 'freight_value']

In [27]:
# Step 10: Behavioral Features

# Recreate observation_items with required columns
observation_items = observation_orders.merge(
    order_items[
        [
            "order_id",
            "order_item_id",
            "product_id",
            "seller_id",
            "price",
            "freight_value"
        ]
    ],
    on="order_id",
    how="left"
)

# Calculate behavioral features
behavioral_features = (
    observation_items
    .groupby("customer_unique_id")
    .agg(
        total_items_obs=("order_item_id", "count"),
        unique_products_obs=("product_id", "nunique"),
        unique_sellers_obs=("seller_id", "nunique")
    )
    .reset_index()
)

# Remove old versions if they already exist
old_columns = [
    "total_items_obs",
    "unique_products_obs",
    "unique_sellers_obs",
    "avg_items_per_order_obs"
]

observation_features = observation_features.drop(
    columns=[col for col in old_columns if col in observation_features.columns]
)

# Merge behavioral features
observation_features = observation_features.merge(
    behavioral_features,
    on="customer_unique_id",
    how="left"
)

# Average items per order
observation_features["avg_items_per_order_obs"] = (
    observation_features["total_items_obs"] /
    observation_features["total_orders_obs"].replace(0, 1)
)

# Check result
print(
    observation_features[
        [
            "customer_unique_id",
            "total_orders_obs",
            "total_items_obs",
            "unique_products_obs",
            "unique_sellers_obs",
            "avg_items_per_order_obs"
        ]
    ].head(10)
)

                 customer_unique_id  total_orders_obs  total_items_obs  \
0  0000f46a3911fa3c0805444483337064                 1                1   
1  0000f6ccb0745a6a4b88665a16c9f078                 1                1   
2  0004aac84e0df4da2b147fca70cf8255                 1                1   
3  0005e1862207bf6ccc02e4228effd9a0                 1                1   
4  0006fdc98a402fceb4eb0ee528f6a8d4                 1                1   
5  00082cbe03e478190aadbea78542e933                 1                1   
6  000a5ad9c4601d2bbdd9ed765d5213b3                 1                1   
7  000bfa1d2f1a41876493be685390d6d3                 1                1   
8  000c8bdb58a29e7115cfc257230fb21b                 1                1   
9  000de6019bb59f34c099a907c151d855                 1                2   

   unique_products_obs  unique_sellers_obs  avg_items_per_order_obs  
0                    1                   1                      1.0  
1                    1                   1   

In [28]:
observation_items.columns.tolist()

['order_id',
 'customer_id',
 'order_status',
 'order_purchase_timestamp',
 'order_approved_at',
 'order_delivered_carrier_date',
 'order_delivered_customer_date',
 'order_estimated_delivery_date',
 'delivery_days',
 'estimated_delivery_days',
 'delivery_delay_days',
 'delivery_performance',
 'customer_unique_id',
 'order_item_id',
 'product_id',
 'seller_id',
 'price',
 'freight_value']

In [30]:
# Step 11: Payment Behavioral Features

payments = pd.read_csv("../data/processed/payments_clean.csv")

# Keep only observation-period orders
observation_payments = observation_orders.merge(
    payments[
        [
            "order_id",
            "payment_type",
            "payment_value",
            "payment_installments"
        ]
    ],
    on="order_id",
    how="left"
)

# Aggregate payment behavior by customer
payment_features = (
    observation_payments
    .groupby("customer_unique_id")
    .agg(
        total_payment_value_obs=("payment_value", "sum"),
        avg_payment_value_obs=("payment_value", "mean"),
        avg_installments_obs=("payment_installments", "mean"),
        payment_count_obs=("payment_type", "count"),
        unique_payment_types_obs=("payment_type", "nunique")
    )
    .reset_index()
)

# Remove old versions if cell is rerun
payment_columns = [
    "total_payment_value_obs",
    "avg_payment_value_obs",
    "avg_installments_obs",
    "payment_count_obs",
    "unique_payment_types_obs"
]

observation_features = observation_features.drop(
    columns=[col for col in payment_columns if col in observation_features.columns]
)

# Merge payment features
observation_features = observation_features.merge(
    payment_features,
    on="customer_unique_id",
    how="left"
)

print(
    observation_features[
        [
            "customer_unique_id",
            "total_payment_value_obs",
            "avg_payment_value_obs",
            "avg_installments_obs",
            "payment_count_obs",
            "unique_payment_types_obs"
        ]
    ].head(10)
)

                 customer_unique_id  total_payment_value_obs  \
0  0000f46a3911fa3c0805444483337064                    86.22   
1  0000f6ccb0745a6a4b88665a16c9f078                    43.62   
2  0004aac84e0df4da2b147fca70cf8255                   196.89   
3  0005e1862207bf6ccc02e4228effd9a0                   150.12   
4  0006fdc98a402fceb4eb0ee528f6a8d4                    29.00   
5  00082cbe03e478190aadbea78542e933                   126.26   
6  000a5ad9c4601d2bbdd9ed765d5213b3                    91.28   
7  000bfa1d2f1a41876493be685390d6d3                    46.85   
8  000c8bdb58a29e7115cfc257230fb21b                    29.00   
9  000de6019bb59f34c099a907c151d855                   257.44   

   avg_payment_value_obs  avg_installments_obs  payment_count_obs  \
0                  86.22                   8.0                  1   
1                  43.62                   4.0                  1   
2                 196.89                   6.0                  1   
3                 1

In [31]:
#Add Review Behavior

In [32]:
# Review Behavioral Features


reviews = pd.read_csv("../data/processed/reviews_clean.csv")

observation_reviews = observation_orders.merge(
    reviews[
        [
            "order_id",
            "review_score"
        ]
    ],
    on="order_id",
    how="left"
)
# Aggregate review behavior by customer
review_features = (
    observation_reviews
    .groupby("customer_unique_id")
    .agg(
        avg_review_score_obs=("review_score", "mean"),
        review_count_obs=("review_score", "count")
    )
    .reset_index()
)

# Remove old versions if cell is rerun
review_columns = [
    "avg_review_score_obs",
    "review_count_obs"
]

observation_features = observation_features.drop(
    columns=[col for col in review_columns if col in observation_features.columns]
)

# Merge review features
observation_features = observation_features.merge(
    review_features,
    on="customer_unique_id",
    how="left"
)

print(
    observation_features[
        [
            "customer_unique_id",
            "avg_review_score_obs",
            "review_count_obs"
        ]
    ].head(10)
)

                 customer_unique_id  avg_review_score_obs  review_count_obs
0  0000f46a3911fa3c0805444483337064                   3.0                 1
1  0000f6ccb0745a6a4b88665a16c9f078                   4.0                 1
2  0004aac84e0df4da2b147fca70cf8255                   5.0                 1
3  0005e1862207bf6ccc02e4228effd9a0                   4.0                 1
4  0006fdc98a402fceb4eb0ee528f6a8d4                   3.0                 1
5  00082cbe03e478190aadbea78542e933                   5.0                 1
6  000a5ad9c4601d2bbdd9ed765d5213b3                   4.0                 1
7  000bfa1d2f1a41876493be685390d6d3                   4.5                 2
8  000c8bdb58a29e7115cfc257230fb21b                   5.0                 1
9  000de6019bb59f34c099a907c151d855                   2.0                 1


In [33]:
#Delivery Behavioral Features

observation_orders["order_purchase_timestamp"] = pd.to_datetime(
    observation_orders["order_purchase_timestamp"]
)

observation_orders["order_delivered_customer_date"] = pd.to_datetime(
    observation_orders["order_delivered_customer_date"]
)
# Calculate delivery days
observation_orders["delivery_days_obs"] = (
    observation_orders["order_delivered_customer_date"]
    - observation_orders["order_purchase_timestamp"]
).dt.total_seconds() / (24 * 60 * 60)

# Aggregate delivery behavior by customer
delivery_features = (
    observation_orders
    .groupby("customer_unique_id")
    .agg(
        avg_delivery_days_obs=("delivery_days_obs", "mean"),
        max_delivery_days_obs=("delivery_days_obs", "max")
    )
    .reset_index()
)

# Remove old versions if cell is rerun
delivery_columns = [
    "avg_delivery_days_obs",
    "max_delivery_days_obs"
]

observation_features = observation_features.drop(
    columns=[col for col in delivery_columns if col in observation_features.columns]
)

# Merge delivery features
observation_features = observation_features.merge(
    delivery_features,
    on="customer_unique_id",
    how="left"
)

print(
    observation_features[
        [
            "customer_unique_id",
            "avg_delivery_days_obs",
            "max_delivery_days_obs"
        ]
    ].head(10)
)


                 customer_unique_id  avg_delivery_days_obs  \
0  0000f46a3911fa3c0805444483337064              25.731759   
1  0000f6ccb0745a6a4b88665a16c9f078              20.037083   
2  0004aac84e0df4da2b147fca70cf8255              13.141134   
3  0005e1862207bf6ccc02e4228effd9a0               4.375648   
4  0006fdc98a402fceb4eb0ee528f6a8d4              16.388646   
5  00082cbe03e478190aadbea78542e933              10.273229   
6  000a5ad9c4601d2bbdd9ed765d5213b3              11.300486   
7  000bfa1d2f1a41876493be685390d6d3              13.960521   
8  000c8bdb58a29e7115cfc257230fb21b              14.113623   
9  000de6019bb59f34c099a907c151d855               4.005660   

   max_delivery_days_obs  
0              25.731759  
1              20.037083  
2              13.141134  
3               4.375648  
4              16.388646  
5              10.273229  
6              11.300486  
7              13.960521  
8              14.113623  
9               4.005660  


In [34]:
# Step 14: Create Final Repeat Purchase ML Dataset

repeat_ml_data = observation_features.merge(
    repeat_data[
        [
            "customer_unique_id",
            "repeat_purchase"
        ]
    ],
    on="customer_unique_id",
    how="left"
)

print("Shape:", repeat_ml_data.shape)

print("\nColumns:")
print(repeat_ml_data.columns.tolist())

print("\nTarget distribution:")
print(repeat_ml_data["repeat_purchase"].value_counts())

print("\nMissing values:")
print(
    repeat_ml_data.isnull().sum()
    .sort_values(ascending=False)
    .head(15)
)

Shape: (44034, 24)

Columns:
['customer_unique_id', 'total_orders_obs', 'first_purchase_date', 'last_purchase_date', 'customer_lifetime_days_obs', 'total_spent_obs', 'total_freight_obs', 'avg_order_value_obs', 'recency_days', 'purchase_frequency_obs', 'total_items_obs', 'unique_products_obs', 'unique_sellers_obs', 'avg_items_per_order_obs', 'total_payment_value_obs', 'avg_payment_value_obs', 'avg_installments_obs', 'payment_count_obs', 'unique_payment_types_obs', 'avg_review_score_obs', 'review_count_obs', 'avg_delivery_days_obs', 'max_delivery_days_obs', 'repeat_purchase']

Target distribution:
repeat_purchase
0    43347
1      687
Name: count, dtype: int64

Missing values:
max_delivery_days_obs         1636
avg_delivery_days_obs         1636
avg_order_value_obs            505
avg_review_score_obs           374
avg_payment_value_obs            1
avg_installments_obs             1
first_purchase_date              0
customer_unique_id               0
total_freight_obs                0
t

In [35]:
# Step 15: Correct Average Order Value

# Calculate total value for each order
order_value_obs = (
    observation_items
    .groupby(["customer_unique_id", "order_id"])
    .agg(
        order_value=("price", "sum")
    )
    .reset_index()
)

# Calculate average order value per customer
correct_aov = (
    order_value_obs
    .groupby("customer_unique_id")
    .agg(
        avg_order_value_obs=("order_value", "mean")
    )
    .reset_index()
)

# Remove the old incorrect feature
repeat_ml_data = repeat_ml_data.drop(
    columns=["avg_order_value_obs"]
)

# Add corrected feature
repeat_ml_data = repeat_ml_data.merge(
    correct_aov,
    on="customer_unique_id",
    how="left"
)

print(
    repeat_ml_data[
        [
            "customer_unique_id",
            "total_orders_obs",
            "total_spent_obs",
            "avg_order_value_obs"
        ]
    ].head(10)
)

                 customer_unique_id  total_orders_obs  total_spent_obs  \
0  0000f46a3911fa3c0805444483337064                 1            69.00   
1  0000f6ccb0745a6a4b88665a16c9f078                 1            25.99   
2  0004aac84e0df4da2b147fca70cf8255                 1           180.00   
3  0005e1862207bf6ccc02e4228effd9a0                 1           135.00   
4  0006fdc98a402fceb4eb0ee528f6a8d4                 1            13.90   
5  00082cbe03e478190aadbea78542e933                 1            79.00   
6  000a5ad9c4601d2bbdd9ed765d5213b3                 1            76.99   
7  000bfa1d2f1a41876493be685390d6d3                 1            35.00   
8  000c8bdb58a29e7115cfc257230fb21b                 1            13.90   
9  000de6019bb59f34c099a907c151d855                 1           229.80   

   avg_order_value_obs  
0                69.00  
1                25.99  
2               180.00  
3               135.00  
4                13.90  
5                79.00  
6         

In [36]:
# Step 16: Prepare Features and Target

X = repeat_ml_data.drop(
    columns=[
        "customer_unique_id",
        "first_purchase_date",
        "last_purchase_date",
        "repeat_purchase"
    ]
)

y = repeat_ml_data["repeat_purchase"]

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nFeatures:")
print(X.columns.tolist())

print("\nTarget distribution:")
print(y.value_counts())

X shape: (44034, 20)
y shape: (44034,)

Features:
['total_orders_obs', 'customer_lifetime_days_obs', 'total_spent_obs', 'total_freight_obs', 'recency_days', 'purchase_frequency_obs', 'total_items_obs', 'unique_products_obs', 'unique_sellers_obs', 'avg_items_per_order_obs', 'total_payment_value_obs', 'avg_payment_value_obs', 'avg_installments_obs', 'payment_count_obs', 'unique_payment_types_obs', 'avg_review_score_obs', 'review_count_obs', 'avg_delivery_days_obs', 'max_delivery_days_obs', 'avg_order_value_obs']

Target distribution:
repeat_purchase
0    43347
1      687
Name: count, dtype: int64


In [37]:
# rain-Test Split

from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(

    X,y,
    test_size = 0.2,
    random_state=42,
    stratify = y
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts())

print("\nTesting target distribution:")
print(y_test.value_counts())

X_train shape: (35227, 20)
X_test shape: (8807, 20)

Training target distribution:
repeat_purchase
0    34677
1      550
Name: count, dtype: int64

Testing target distribution:
repeat_purchase
0    8670
1     137
Name: count, dtype: int64


In [39]:
# Step 18: Preprocessing

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Processed X_train shape:", X_train_processed.shape)
print("Processed X_test shape:", X_test_processed.shape)

print("Missing values after preprocessing:")
print("X_train:", np.isnan(X_train_processed).sum())
print("X_test:", np.isnan(X_test_processed).sum())

Processed X_train shape: (35227, 20)
Processed X_test shape: (8807, 20)
Missing values after preprocessing:
X_train: 0
X_test: 0


In [40]:
#  Logistic Regression
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(
    class_weight = "balanced",
    max_iter = 1000,
    random_state = 42
)

lr_model.fit(X_train_processed, y_train)

y_pred_lr = lr_model.predict(X_test_processed)
y_prob_lr = lr_model.predict_proba(X_test_processed)[:,1]

print("Logistic Regression trained successfully.")


Logistic Regression trained successfully.


In [42]:
#  Evaluate Logistic Regression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

lr_accuracy = accuracy_score(y_test, y_pred_lr)
lr_precision = precision_score(y_test,y_pred_lr,zero_division=0)
lr_recall = recall_score(y_test, y_pred_lr, zero_division=0)
lr_f1 = f1_score(y_test, y_pred_lr, zero_division=0)
lr_roc_auc = roc_auc_score(y_test, y_prob_lr)
lr_pr_auc = average_precision_score(y_test, y_prob_lr)

print("Logistic Regression Results")
print("---------------------------")
print("Accuracy :", round(lr_accuracy, 4))
print("Precision:", round(lr_precision, 4))
print("Recall   :", round(lr_recall, 4))
print("F1 Score :", round(lr_f1, 4))
print("ROC-AUC  :", round(lr_roc_auc, 4))
print("PR-AUC   :", round(lr_pr_auc, 4))


Logistic Regression Results
---------------------------
Accuracy : 0.6399
Precision: 0.0187
Recall   : 0.4307
F1 Score : 0.0359
ROC-AUC  : 0.567
PR-AUC   : 0.0374


In [43]:
# Random Forest
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth = 15,
    min_samples_split=5,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state = 42,
    n_jobs = -1
)

rf_model.fit(X_train_processed, y_train)

y_pred_rf = rf_model.predict(X_test_processed)
y_prob_rf = rf_model.predict_proba(X_test_processed)[:,1]
print("Random Forest trained successfully")

Random Forest trained successfully


In [45]:
# Evaluate Random Forest

rf_accuracy = accuracy_score(y_test,y_pred_rf)
rf_precision = precision_score(y_test, y_pred_rf, zero_division =0)
rf_recall = recall_score(y_test, y_pred_rf, zero_division=0)
rf_f1 = f1_score(y_test,y_pred_rf, zero_division=0)
rf_roc_auc = roc_auc_score(y_test, y_prob_rf)
rf_pr_auc = average_precision_score(y_test, y_prob_rf)

print("Random Forest Results")
print("---------------------")
print("Accuracy :", round(rf_accuracy, 4))
print("Precision:", round(rf_precision, 4))
print("Recall   :", round(rf_recall, 4))
print("F1 Score :", round(rf_f1, 4))
print("ROC-AUC  :", round(rf_roc_auc, 4))
print("PR-AUC   :", round(rf_pr_auc, 4))


Random Forest Results
---------------------
Accuracy : 0.9508
Precision: 0.0132
Recall   : 0.0292
F1 Score : 0.0181
ROC-AUC  : 0.4879
PR-AUC   : 0.0187


In [46]:
# Gradient Boosting

from sklearn.ensemble import GradientBoostingClassifier

gb_model = GradientBoostingClassifier(
    n_estimators = 200,
    learning_rate = 0.05,
    max_depth = 3,
    random_state = 42
)
gb_model.fit(X_train_processed, y_train)

y_pred_gb = gb_model.predict(X_test_processed)
y_prob_gb = gb_model.predict_proba(X_test_processed)[:,1]
print("Gradient Boosting trained successfully.")


Gradient Boosting trained successfully.


In [47]:
# Step 24: Evaluate Gradient Boosting

gb_accuracy = accuracy_score(y_test, y_pred_gb)
gb_precision = precision_score(y_test, y_pred_gb, zero_division=0)
gb_recall = recall_score(y_test, y_pred_gb, zero_division=0)
gb_f1 = f1_score(y_test, y_pred_gb, zero_division=0)
gb_roc_auc = roc_auc_score(y_test, y_prob_gb)
gb_pr_auc = average_precision_score(y_test, y_prob_gb)

print("Gradient Boosting Results")
print("-------------------------")
print("Accuracy :", round(gb_accuracy, 4))
print("Precision:", round(gb_precision, 4))
print("Recall   :", round(gb_recall, 4))
print("F1 Score :", round(gb_f1, 4))
print("ROC-AUC  :", round(gb_roc_auc, 4))
print("PR-AUC   :", round(gb_pr_auc, 4))

Gradient Boosting Results
-------------------------
Accuracy : 0.9842
Precision: 0.0
Recall   : 0.0
F1 Score : 0.0
ROC-AUC  : 0.5303
PR-AUC   : 0.02


In [50]:
#  Model Comparison

model_comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest",
        "Gradient Boosting"
    ],
    "Accuracy":[
        lr_accuracy,
        rf_accuracy,
        gb_accuracy

    ],
    "Precision": [
        lr_precision,
        rf_precision,
        gb_precision
    ],
    "Recall": [
        lr_recall,
        rf_recall,
        gb_recall
    ],
    "F1 Score": [
        lr_f1,
        rf_f1,
        gb_f1
    ],
    "ROC-AUC":[
    lr_pr_auc,
    rf_pr_auc,
    gb_pr_auc
    ],
    "PR-AUC": [
        lr_pr_auc,
        rf_pr_auc,
        gb_pr_auc
    ]
})

print(model_comparison.round(4))









                 Model  Accuracy  Precision  Recall  F1 Score  ROC-AUC  PR-AUC
0  Logistic Regression    0.6399     0.0187  0.4307    0.0359   0.0374  0.0374
1        Random Forest    0.9508     0.0132  0.0292    0.0181   0.0187  0.0187
2    Gradient Boosting    0.9842     0.0000  0.0000    0.0000   0.0200  0.0200


In [49]:
model_comparison.to_csv(
    "../data/processed/repeat_purchase_model_comparison.csv",
    index=False
)

print("Model comparison saved successfully.")

Model comparison saved successfully.


In [51]:
model_comparison.to_csv(
    "../data/processed/repeat_purchase_model_comparison.csv",
    index=False
)

print("Corrected model comparison saved successfully.")

Corrected model comparison saved successfully.


In [54]:
from sklearn.metrics import precision_score, recall_score, f1_score

threshold_results = []

for threshold in np.arange(0.10, 0.91, 0.05):

    y_pred_threshold = (y_prob_lr >= threshold).astype(int)

    precision = precision_score(
        y_test,
        y_pred_threshold,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        y_pred_threshold,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        y_pred_threshold,
        zero_division=0
    )

    threshold_results.append([
        threshold,
        precision,
        recall,
        f1
    ])

threshold_analysis = pd.DataFrame(
    threshold_results,
    columns=[
        "Threshold",
        "Precision",
        "Recall",
        "F1 Score"
    ]
)

print(threshold_analysis.round(4))

    Threshold  Precision  Recall  F1 Score
0        0.10     0.0156  1.0000    0.0307
1        0.15     0.0156  1.0000    0.0308
2        0.20     0.0157  1.0000    0.0309
3        0.25     0.0158  1.0000    0.0311
4        0.30     0.0160  1.0000    0.0315
5        0.35     0.0161  0.9635    0.0317
6        0.40     0.0165  0.8905    0.0324
7        0.45     0.0176  0.7518    0.0344
8        0.50     0.0187  0.4307    0.0359
9        0.55     0.0185  0.1679    0.0334
10       0.60     0.0373  0.1387    0.0587
11       0.65     0.0528  0.0949    0.0679
12       0.70     0.0602  0.0730    0.0660
13       0.75     0.0783  0.0657    0.0714
14       0.80     0.0896  0.0438    0.0588
15       0.85     0.0909  0.0219    0.0353
16       0.90     0.1538  0.0146    0.0267


In [55]:
# Evaluate Logistic Regression at threshold 0.75

best_threshold = 0.75

y_pred_lr_tuned = (
    y_prob_lr >= best_threshold
).astype(int)

tuned_precision = precision_score(
    y_test,
    y_pred_lr_tuned,
    zero_division = 0
)
tuned_recall =recall_score(
    y_test,
    y_pred_lr_tuned,
    zero_division = 0
)
tuned_f1 = f1_score(
    y_test,
    y_pred_lr_tuned,
    zero_division =0
)

print("Threshold:", best_threshold)
print("Precision:", round(tuned_precision, 4))
print("Recall:", round(tuned_recall, 4))
print("F1 Score:", round(tuned_f1, 4))

Threshold: 0.75
Precision: 0.0783
Recall: 0.0657
F1 Score: 0.0714


In [56]:
threshold_analysis.to_csv(
    "../data/processed/repeat_purchase_threshold_analysis.csv",
    index=False
)

print("Threshold analysis saved successfully.")

Threshold analysis saved successfully.


In [57]:
repeat_predictions = pd.DataFrame({
    "customer_unique_id": repeat_ml_data["customer_unique_id"].iloc[X_test.index],
    "actual_repeat_purchase": y_test.values,
    "repeat_purchase_probability": y_prob_lr,
    "predicted_repeat_purchase": y_pred_lr_tuned
})

print(repeat_predictions.head())
print("\nShape:", repeat_predictions.shape)

                     customer_unique_id  actual_repeat_purchase  \
19006  6e5e2b5c13a9c5f57d79b501d6a66c39                       0   
27693  a112f7591e1fede1435eed37d2a3d64b                       0   
6482   258c1c4ccff86dd5d665464fa6b456aa                       0   
36026  d10948d77bc5f66fa786d9e3a5c459a1                       0   
28681  a6b57cbf3b29e6fcfb86cd50c2c77751                       0   

       repeat_purchase_probability  predicted_repeat_purchase  
19006                     0.437879                          0  
27693                     0.485253                          0  
6482                      0.324755                          0  
36026                     0.551981                          0  
28681                     0.334355                          0  

Shape: (8807, 4)


In [58]:
repeat_predictions.to_csv(
    "../data/processed/repeat_purchase_predictions.csv",
    index=False
)

print("Repeat purchase predictions saved successfully.")

Repeat purchase predictions saved successfully.


In [59]:
repeat_predictions.to_csv(
    "../data/processed/repeat_purchase_predictions.csv",
    index=False
)

print("Repeat purchase predictions saved successfully.")

Repeat purchase predictions saved successfully.


In [60]:
print("===== PART 09 FINAL VALIDATION =====")

print("\n1. Dataset:")
print("Repeat ML data shape:", repeat_ml_data.shape)

print("\n2. Target distribution:")
print(repeat_ml_data["repeat_purchase"].value_counts())

print("\n3. Model comparison:")
print(model_comparison.round(4))

print("\n4. Best threshold:")
print("Threshold:", best_threshold)
print("Precision:", round(tuned_precision, 4))
print("Recall:", round(tuned_recall, 4))
print("F1 Score:", round(tuned_f1, 4))

print("\n5. Predictions:")
print("Prediction file shape:", repeat_predictions.shape)

print("\n6. Saved files:")
print("✓ repeat_purchase_model_comparison.csv")
print("✓ repeat_purchase_threshold_analysis.csv")
print("✓ repeat_purchase_predictions.csv")

print("\n===== PART 09 VALIDATION COMPLETE =====")

===== PART 09 FINAL VALIDATION =====

1. Dataset:
Repeat ML data shape: (44034, 24)

2. Target distribution:
repeat_purchase
0    43347
1      687
Name: count, dtype: int64

3. Model comparison:
                 Model  Accuracy  Precision  Recall  F1 Score  ROC-AUC  PR-AUC
0  Logistic Regression    0.6399     0.0187  0.4307    0.0359   0.0374  0.0374
1        Random Forest    0.9508     0.0132  0.0292    0.0181   0.0187  0.0187
2    Gradient Boosting    0.9842     0.0000  0.0000    0.0000   0.0200  0.0200

4. Best threshold:
Threshold: 0.75
Precision: 0.0783
Recall: 0.0657
F1 Score: 0.0714

5. Predictions:
Prediction file shape: (8807, 4)

6. Saved files:
✓ repeat_purchase_model_comparison.csv
✓ repeat_purchase_threshold_analysis.csv
✓ repeat_purchase_predictions.csv

===== PART 09 VALIDATION COMPLETE =====
